# OEC AudioGen — Gradio sound-effects worker

Kernel: phase1. Backend: AudioCraft 1.3.0 / Torch 2.1.0 + CUDA 11.8. This notebook uses the existing CLI in offline/cache-only mode; no package/model downloads are attempted.


## Environment and safe smoke settings

- Jupyter kernel: phase1 (AudioCraft 1.3.0, Torch 2.1.0, CUDA 11.8); the same kernel invokes the existing AudioGen CLI.
- The UI runs one subprocess at a time, uses --offline and --scratch-output, and writes only a short test artifact under /tmp.
- The CLI must select an unoccupied A100. If all GPUs have existing compute owners or the checkpoint is not cached, the request fails closed; do not kill unrelated processes or trigger a blind download.

In [ ]:
from pathlib import Path
import os, subprocess, sys, tempfile, uuid
import gradio as gr
ROOT = Path.cwd().resolve()
if not (ROOT / "pipeline_audiogen.py").is_file():
    ROOT = Path(os.environ.get("OEC_REPO_ROOT", str(ROOT))).expanduser().resolve()
if not (ROOT / "pipeline_audiogen.py").is_file():
    raise RuntimeError("Run in the OEC repo root or set OEC_REPO_ROOT.")
def generate_sound_effect(prompt, duration, seed):
    if not isinstance(prompt, str) or not prompt.strip():
        raise gr.Error("Enter a sound-effect description.")
    if not 1 <= int(duration) <= 10:
        raise gr.Error("Duration must be 1–10 seconds.")
    output = Path("/tmp") / f"oec-audiogen-{uuid.uuid4().hex}.wav"
    cmd = [sys.executable, str(ROOT / "pipeline_audiogen.py"), "--mode", "generate", "--offline", "--scratch-output", "--output", str(output), "--prompt", prompt.strip(), "--duration", str(int(duration)), "--seed", str(int(seed))]
    result = subprocess.run(cmd, cwd=ROOT, text=True, capture_output=True, timeout=900, check=False)
    log = (result.stdout + "\n" + result.stderr)[-8000:]
    if result.returncode: raise gr.Error("AudioGen failed in offline mode.\n" + log)
    if not output.is_file() or output.stat().st_size == 0: raise gr.Error("Missing or empty output WAV.")
    info = output.with_suffix(".info.json")
    return str(output), info.read_text(encoding="utf-8") if info.is_file() else log
demo = gr.Interface(fn=generate_sound_effect, inputs=[gr.Textbox(label="Sound effect prompt", lines=3), gr.Slider(1, 10, value=3, step=1), gr.Number(value=42, precision=0)], outputs=[gr.Audio(type="filepath"), gr.Textbox(lines=10)], title="OEC AudioGen")
if os.environ.get("OEC_GRADIO_HEADLESS_SMOKE") != "1":
    demo.queue(max_size=1).launch(server_name="127.0.0.1", server_port=7901, share=False, inline=False)
